# Notebook 05: Hyperparameter Tuning & Model Evaluation

**Objective:** Optimize models and perform comprehensive evaluation

**Techniques:**
- TrainValidationSplit for hyperparameter tuning
- Grid search with reduced parameters (memory optimized)
- Feature importance analysis
- Model evaluation and comparison

---

## 1. Setup

In [1]:
!pip install -q pyspark==3.4.0 pyarrow pandas matplotlib seaborn
print("✓ Libraries installed")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✓ Libraries installed


In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from pyspark.sql import SparkSession
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, abs as spark_abs

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("NYC_Taxi_Tuning") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print(f"✓ Spark {spark.version}")

26/03/01 12:20:59 WARN Utils: Your hostname, durga-Aspire-A715-79G resolves to a loopback address: 127.0.1.1; using 10.109.1.195 instead (on interface wlp0s20f3)
26/03/01 12:20:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 12:21:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/01 12:21:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/01 12:21:00 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/01 12:21:00 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/01 12:21:00 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/03/01 12:21:00 WARN Utils: Service 'SparkUI' could not bind on port 4044. Atte

✓ Spark 3.4.0


## 2. Load Data

In [4]:
gold_path = "data/gold/nyc_taxi_features"
df_gold = spark.read.parquet(gold_path)

# Split data
train_data, test_data = df_gold.randomSplit([0.8, 0.2], seed=42)
train_data.cache()
test_data.cache()

print(f"✓ Data loaded")
print(f"  Training: {train_data.count():,}")
print(f"  Testing: {test_data.count():,}")

✓ Data loaded


  Training: 7,017,522


  Testing: 1,754,745


## 3. Hyperparameter Tuning - Random Forest

In [5]:
print("Tuning Random Forest...")
start_time = time.time()

# Define model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    seed=42
)

# Reduced parameter grid for memory efficiency
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

# Evaluator
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

# Use TrainValidationSplit instead of CrossValidator (faster, less memory)
tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42
)

# Fit
rf_model = tvs.fit(train_data)

tuning_time = time.time() - start_time

print(f"✓ Random Forest tuned in {tuning_time:.2f}s")
print(f"  Best RMSE: ${min(rf_model.validationMetrics):.2f}")

Tuning Random Forest...


26/03/01 12:21:48 WARN MemoryStore: Not enough space to cache rdd_50_10 in memory! (computed 328.8 MiB so far)
26/03/01 12:21:49 WARN BlockManager: Persisting block rdd_50_10 to disk instead.
26/03/01 12:21:49 WARN MemoryStore: Not enough space to cache rdd_50_6 in memory! (computed 328.8 MiB so far)
26/03/01 12:21:49 WARN BlockManager: Persisting block rdd_50_6 to disk instead.
26/03/01 12:22:11 WARN MemoryStore: Not enough space to cache rdd_101_2 in memory! (computed 328.8 MiB so far)
26/03/01 12:22:11 WARN MemoryStore: Not enough space to cache rdd_101_10 in memory! (computed 328.8 MiB so far)
26/03/01 12:22:11 WARN BlockManager: Persisting block rdd_101_10 to disk instead.
26/03/01 12:22:11 WARN BlockManager: Persisting block rdd_101_2 to disk instead.
26/03/01 12:22:39 WARN DAGScheduler: Broadcasting large task binary with size 1598.6 KiB
26/03/01 12:22:49 WARN MemoryStore: Not enough space to cache rdd_161_2 in memory! (computed 339.5 MiB so far)
26/03/01 12:22:49 WARN BlockMana

✓ Random Forest tuned in 266.95s
  Best RMSE: $3.36


## 4. Hyperparameter Tuning - GBT

In [ ]:
print("Tuning Gradient Boosted Trees...")
start_time = time.time()

# Define model
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    seed=42
)

# Reduced parameter grid
paramGrid_gbt = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [10, 20]) \
    .addGrid(gbt.maxDepth, [3, 5]) \
    .build()

# Train-validation split (faster than CV)
tvs_gbt = TrainValidationSplit(
    estimator=gbt,
    estimatorParamMaps=paramGrid_gbt,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42
)

# Fit
gbt_model = tvs_gbt.fit(train_data)

tuning_time = time.time() - start_time

print(f"✓ GBT tuned in {tuning_time:.2f}s")
print(f"  Best RMSE: ${min(gbt_model.validationMetrics):.2f}")

Tuning Gradient Boosted Trees...


26/03/01 12:26:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


## 5. Evaluate Best Models

In [ ]:
# Get best models
best_rf = rf_model.bestModel
best_gbt = gbt_model.bestModel

# Evaluate on test data
rf_predictions = best_rf.transform(test_data)
gbt_predictions = best_gbt.transform(test_data)

# Calculate metrics
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
evaluator_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")

rf_rmse = evaluator_rmse.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)
rf_mae = evaluator_mae.evaluate(rf_predictions)

gbt_rmse = evaluator_rmse.evaluate(gbt_predictions)
gbt_r2 = evaluator_r2.evaluate(gbt_predictions)
gbt_mae = evaluator_mae.evaluate(gbt_predictions)

print("\n" + "="*60)
print("TUNED MODEL RESULTS")
print("="*60)
print(f"\nRandom Forest:")
print(f"  RMSE: ${rf_rmse:.2f}")
print(f"  R²: {rf_r2:.4f}")
print(f"  MAE: ${rf_mae:.2f}")
print(f"\nGradient Boosted Trees:")
print(f"  RMSE: ${gbt_rmse:.2f}")
print(f"  R²: {gbt_r2:.4f}")
print(f"  MAE: ${gbt_mae:.2f}")
print("="*60)

## 6. Stability Analysis & Statistical Significance

**Requirement:** "Stability indicates how a machine learning algorithm output is changed with small perturbations to its inputs."

### 6.1 Stability Test (Perturbation Analysis)
We will add small random noise to the features and measure how much the predictions deviate.

In [ ]:
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import rand, udf
from pyspark.ml.linalg import Vectors, VectorUDT

print("Running Stability Analysis...")

# Get a sample
stability_sample = test_data.sample(fraction=0.1, seed=42)

# Make predictions on original
orig_preds = best_rf.transform(stability_sample).select("label", "prediction", "features")

# Define perturbation function
def perturb_vector(v, epsilon=0.01):
    arr = v.toArray()
    noise = np.random.normal(0, epsilon, len(arr))
    return Vectors.dense(arr + noise)

perturb_udf = udf(lambda x: perturb_vector(x), VectorUDT())

# Apply perturbation
perturbed_df = stability_sample.withColumn("features_perturbed", perturb_udf(col("features")))

# Predict on perturbed
perturbed_preds = best_rf.transform(perturbed_df.withColumn("features", col("features_perturbed"))) \
    .withColumnRenamed("prediction", "prediction_perturbed") \
    .select("prediction_perturbed")

# Compare predictions
rdd_orig = orig_preds.select("prediction").rdd.map(lambda r: r[0])
rdd_pert = perturbed_preds.rdd.map(lambda r: r[0])

diffs = rdd_orig.zip(rdd_pert).map(lambda x: abs(x[0] - x[1])).collect()

avg_stability_diff = np.mean(diffs)
print(f"✓ Stability Analysis Complete")
print(f"  Average Prediction Change with Noise: ${avg_stability_diff:.4f}")
print(f"  Interpretation: Lower is more stable. Change < $0.10 implies high stability.")

### 6.2 Statistical Significance (Bootstrap Confidence Intervals)

Calculate 95% Confidence Interval for RMSE using Bootstrapping.

In [ ]:
print("Calculating 95% Confidence Intervals (Bootstrap)...")

# Collect residuals (difference between prediction and label)
# We use a sample to keep it computationally feasible
residuals = rf_predictions.select(col("label") - col("prediction")).rdd.map(lambda r: r[0]).sample(False, 0.1, 42).collect()

n_bootstrap = 1000
bootstrap_rmses = []

for _ in range(n_bootstrap):
    # Resample with replacement
    sample_residuals = np.random.choice(residuals, size=len(residuals), replace=True)
    rmse = np.sqrt(np.mean(sample_residuals**2))
    bootstrap_rmses.append(rmse)

# Calculate percentiles
ci_lower = np.percentile(bootstrap_rmses, 2.5)
ci_upper = np.percentile(bootstrap_rmses, 97.5)

print(f"✓ Bootstrap Analysis ({n_bootstrap} iterations)")
print(f"  RMSE Mean: ${np.mean(bootstrap_rmses):.2f}")
print(f"  95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")

## 7. Feature Importance

In [ ]:
# Get feature importances from Random Forest
feature_names = [
    'trip_distance',
    'passenger_count', 
    'pickup_hour',
    'pickup_day_of_week',
    'pickup_month',
    'is_rush_hour',
    'is_weekend',
    'is_airport_trip',
    'avg_speed_mph',
    'is_credit_card',
    'RatecodeID',
    'PULocationID',
    'DOLocationID'
]

importances = best_rf.featureImportances.toArray()

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("\nFeature Importances (Random Forest):")
print(feature_importance_df.to_string(index=False))

# Plot
plt.figure(figsize=(10, 8))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importances - Random Forest')
plt.tight_layout()
plt.savefig('results/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature importance plot saved to 'results/feature_importance.png'")

## 8. Save Tuned Models

In [ ]:
from pathlib import Path

# Create directories
Path("models/tuned").mkdir(parents=True, exist_ok=True)
Path("results").mkdir(exist_ok=True)

# Save models
best_rf.write().overwrite().save("models/tuned/random_forest")
best_gbt.write().overwrite().save("models/tuned/gbt")

# Save results
results_df = pd.DataFrame({
    'Model': ['Random Forest (Tuned)', 'GBT (Tuned)'],
    'RMSE': [rf_rmse, gbt_rmse],
    'R2': [rf_r2, gbt_r2],
    'MAE': [rf_mae, gbt_mae]
})

results_df.to_csv('results/tuned_model_results.csv', index=False)

print("✓ Tuned models saved to 'models/tuned/'")
print("✓ Results saved to 'results/tuned_model_results.csv'")

## Summary

This notebook completed:
- Hyperparameter tuning for Random Forest and GBT using TrainValidationSplit
- Model evaluation on test data
- Stability analysis with perturbation testing
- Statistical significance with bootstrap confidence intervals
- Feature importance analysis
- Saved tuned models and results

**Next:** Notebook 06 - Scalability Analysis